# WeightedKgBlend — Step 1: Prepare Splits
Merges MIND graph files and creates 5 random 80/10/10 indication splits.
Outputs saved to `/kaggle/working/splits/` for use by model kernels.


In [ ]:

# ── Find MIND files (handles any subdirectory structure) ───────────────────
from pathlib import Path

INPUT = Path('/kaggle/input')
WORK  = Path('/kaggle/working')

# Find train.txt — largest .txt file in input
txts = sorted(INPUT.rglob('train.txt'), key=lambda f: f.stat().st_size, reverse=True)
if not txts:
    raise FileNotFoundError(f"train.txt not found under {INPUT}. Files found: {list(INPUT.rglob('*'))[:20]}")

TRAIN = txts[0]
BASE  = TRAIN.parent
TEST  = BASE / 'test.txt'
VALID = BASE / 'valid.txt'

print(f"Dataset root : {BASE}")
print(f"train.txt    : {TRAIN.stat().st_size/1024**2:.0f} MB")
print(f"test.txt     : {TEST.stat().st_size/1024:.0f} KB  exists={TEST.exists()}")
print(f"valid.txt    : {VALID.stat().st_size/1024:.0f} KB  exists={VALID.exists()}")


In [ ]:

# ── Merge into single graph ─────────────────────────────────────────────────
import shutil, pandas as pd

MIND = WORK / 'mind.tsv'
print('Merging train + test + valid...')
with open(MIND, 'wb') as out:
    for p in [TRAIN, TEST, VALID]:
        if p.exists():
            with open(p, 'rb') as f:
                shutil.copyfileobj(f, out)

print(f'mind.tsv: {MIND.stat().st_size/1024**2:.1f} MB')

full = pd.read_csv(MIND, sep='\t', header=None, names=['head','relation','tail'])
print(f'Total triples  : {len(full):,}')
print(f'Unique relations: {full.relation.nunique()}')
print()
print('Top relations:')
print(full.relation.value_counts().head(15).to_string())


In [ ]:

# ── Config ──────────────────────────────────────────────────────────────────
INDICATION_REL = 'indication'   # verify from top relations above
N_SPLITS       = 5
RANDOM_SEED    = 42
TRAIN_RATIO    = 0.80
TEST_RATIO     = 0.10
VALID_RATIO    = 0.10

print(f'Indication relation : {INDICATION_REL}')
print(f'N splits            : {N_SPLITS}')
print(f'Split ratio         : {TRAIN_RATIO}/{TEST_RATIO}/{VALID_RATIO}')


In [ ]:

# ── Create 5 splits ─────────────────────────────────────────────────────────
import numpy as np, random

SPLITS = WORK / 'splits'
SPLITS.mkdir(exist_ok=True)

ind     = full[full.relation == INDICATION_REL].reset_index(drop=True)
non_ind = full[full.relation != INDICATION_REL]

print(f'Indication triples : {len(ind):,}')
print(f'Non-indication     : {len(non_ind):,}')
print()

if len(ind) == 0:
    raise ValueError(f'No triples found for relation "{INDICATION_REL}". '
                     f'Check INDICATION_REL above — available: {full.relation.value_counts().head(10).to_dict()}')

all_entities = pd.Series(pd.unique(full[['head','tail']].values.ravel()))
seeds = [RANDOM_SEED + i * 100 for i in range(N_SPLITS)]

for i, seed in enumerate(seeds):
    sl = SPLITS / f'slice_{i}'
    sl.mkdir(exist_ok=True)

    idx = list(range(len(ind)))
    random.seed(seed)
    random.shuffle(idx)

    n      = len(idx)
    n_test = int(n * TEST_RATIO)
    n_val  = int(n * VALID_RATIO)

    ind_test  = ind.iloc[idx[:n_test]]
    ind_valid = ind.iloc[idx[n_test:n_test + n_val]]
    ind_train = ind.iloc[idx[n_test + n_val:]]

    # KGE training = full non-indication graph + indication train triples
    kge_train = pd.concat([non_ind, ind_train], ignore_index=True)

    kge_train.to_csv(sl / 'kge_train.tsv', sep='\t', index=False, header=False)
    ind_train.to_csv(sl / 'ind_train.tsv',  sep='\t', index=False, header=False)
    ind_test.to_csv( sl / 'ind_test.tsv',   sep='\t', index=False, header=False)
    ind_valid.to_csv(sl / 'ind_valid.tsv',  sep='\t', index=False, header=False)
    all_entities.to_csv(sl / 'entities.txt', index=False, header=False)

    print(f'slice_{i} (seed={seed}):  train={len(ind_train):,}  test={len(ind_test):,}  valid={len(ind_valid):,}  kge_train={len(kge_train):,}')

print()
print('All splits saved to /kaggle/working/splits/')


In [ ]:

# ── Sanity check ─────────────────────────────────────────────────────────────
for i in range(N_SPLITS):
    sl    = SPLITS / f'slice_{i}'
    files = [f.name for f in sl.iterdir()]
    sizes = {f: (sl/f).stat().st_size for f in files}
    print(f'slice_{i}: {sizes}')
